# PD 모델링
## Layer 2 — 정책 입력값을 신뢰할 수 있게 만드는 첫 단계

## 목적

PD 모델링의 목적은 `02_eda_policy_design.ipynb`에서 정의한 DTI, CIR, LTV를 실제 위험도 분리 기준으로 연결하는 데 있습니다. 분석 결과, 이 변수들만으로는 고객의 부도 위험을 충분히 세밀하게 구분하기 어려웠고, 특히 DTI는 단독 분리력이 제한적이었습니다. 따라서 승인과 한도 정책의 1차 기준은 개별 재무 비율이 아니라 부도 확률(PD)로 두고, LTV는 1.2 전후에서 부도율 차이가 관찰된 독립적 위험 신호로, DTI는 PD가 경계 구간일 때만 보조적으로 확인하는 조건으로, CIR은 상환기간과 DTI 상한을 역산해 정한 한도 상한 제약으로 각각 다른 역할을 맡기는 방향이 타당하다고 판단했습니다.

따라서 이 노트북의 목적은 **PD를 신용 리스크 기반 한도 정책 설계에 활용 가능한 입력값**으로 만드는 데 있습니다. 즉, 모델 성능 자체보다 ‘누구에게 얼마를 빌려줄 것인가’라는 의사결정에 연결되는지 확인하는 것이 핵심입니다. 이를 위해 다음의 두 가지 축으로 검증합니다.

1. 단순 규칙이나 베이스라인 대비 분리력 개선 여부 (AUC, KS)
2. 성능 개선이 일반화 가능 여부 (결측 처리 방식이 정보를 보존하는가, 튜닝이 성능만이 아니라 안정성도 고려했는가)

두 가지 기준을 충족한 모델을 선정한 뒤에는 calibration을 통해 PD를 실제 확률값으로 보정하고, 이후 정책 시뮬레이션의 EL 계산과 라벨링 입력값으로 활용합니다.

## Step 1. 단순 규칙 기반 하한선 확인

본격적인 모델링에 앞서 단순 규칙만으로 설명 가능한 기준선을 먼저 확인하고, 이후 모델이 제공하는 추가 신호와 의사결정 가치를 검증합니다. 규칙은 검증된 신호 강도가 다른 세 변수 (EXT_SOURCE, LTV, DTI)로 구성했습니다.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import ks_2samp
import warnings
warnings.filterwarnings('ignore')

val_lr = pd.read_csv('../data/val_lr.csv')
y_val = val_lr['TARGET']

def evaluate(y_true, y_score, name):
    scores_pos = y_score[y_true == 1]
    scores_neg = y_score[y_true == 0]
    ks_stat, _ = ks_2samp(scores_pos, scores_neg)
    auc    = roc_auc_score(y_true, y_score)
    pr_auc = average_precision_score(y_true, y_score)
    print(f"{name:35s} AUC: {auc:.4f}  PR-AUC: {pr_auc:.4f}  KS: {ks_stat:.4f}")
    return {'Model': name, 'AUC': round(auc,4), 'PR_AUC': round(pr_auc,4), 'KS': round(ks_stat,4)}

results = []

# Rule 1: EXT_SOURCE 평균 — 점수가 낮을수록 부도 가능성 높음
# 모델 성능과 FN 탐지 등에 중요한 영향을 미치는 변수 기준 단순 규칙 설정
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
val_ext = val_lr[ext_cols].copy()
val_ext['EXT_SOURCE_1'] = val_ext['EXT_SOURCE_1'].replace(-1, np.nan)
rule1_score = (1 - val_ext.mean(axis=1)).fillna(0.5)
results.append(evaluate(y_val, rule1_score, "Rule 1: EXT_SOURCE 평균 (강한 신호)"))

# Rule 2: LTV — 1.2를 기점으로 값이 클수록 부도율 증가
rule2_score = val_lr['LTV'].fillna(val_lr['LTV'].median())
rule2_score = (rule2_score / rule2_score.quantile(0.99)).clip(0, 1)  # 0~1 정규화
results.append(evaluate(y_val, rule2_score, "Rule 2: LTV 단일 (독립 신호)"))

# Rule 3: DTI — 단조 관계 부재 재확인
rule3_score = val_lr['DTI'].fillna(val_lr['DTI'].median()).clip(0, 1)
results.append(evaluate(y_val, rule3_score, "Rule 3: DTI 단일 (약한 신호)"))

Rule 1: EXT_SOURCE 평균 (강한 신호)       AUC: 0.7166  PR-AUC: 0.1992  KS: 0.3169
Rule 2: LTV 단일 (독립 신호)              AUC: 0.5719  PR-AUC: 0.1046  KS: 0.1275
Rule 3: DTI 단일 (약한 신호)              AUC: 0.5185  PR-AUC: 0.0846  KS: 0.0374


LTV는 연속적인 크기 자체보다 1.2라는 임계값을 넘는지 여부에서 실질적인 의미가 나타났고, DTI는 AUC 0.5185로 랜덤 수준에 가까워 단독 예측력은 제한적이었습니다. 따라서 변수 하나만으로 설명되는 단순 규칙보다, 최소한 EXT_SOURCE 단일 규칙의 성능인 AUC 0.7166, KS 0.3169를 넘어서는 수준의 분리력을 보여야 여러 피처를 결합한 모델을 도입할 정당성이 생긴다고 판단했습니다. 결국 이 단계의 목적은 개별 변수의 신호 강도를 확인하고, 이후 다변량 모델이 실제로 추가 가치를 만드는지 검증하는 데 있습니다.

## Step 2. Logistic Regression 베이스라인

신용평가(credit scoring)에서는 해석 가능성이 중요한 만큼 LR 기반 스코어카드가 표준적으로 활용됩니다. 심사역이 결과를 이해하고 규제 대응 시 판단 근거를 설명해야 하는 환경을 고려해, 본 프로젝트에서는 LR을 설명 가능성과 실무 적합성을 갖춘 베이스라인으로 설정했습니다.

LR은 선형 결합 기반 모델이므로 계수 해석을 유지할 수 있도록 numerical 피처를 표준화했고, 클래스 불균형으로 인한 편향을 완화하기 위해 `classweight`를 적용했습니다. 학습 데이터는 01_data_mart.ipynb에서 전처리한 `train_lr.csv`를 사용했으며, `EXTSOURCE_1`은 -1, 나머지 결측치는 Train 중앙값으로 대체했습니다.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

train_lr = pd.read_csv('../data/train_lr.csv')

numeric_features = [
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'FLAG_EXT_SOURCE_MISSING',
    'AGE', 'EMPLOYMENT_YEARS', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH',
    'DTI', 'CIR', 'LTV', 'REPAYMENT_MONTHS',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT',
    'PAYMENT_RATIO', 'LATE_PAYMENT_COUNT', 'LATE_DAYS_AVG',
    'BUREAU_ACTIVE_COUNT', 'BUREAU_OVERDUE_COUNT', 'BUREAU_LOAN_COUNT', 'BUREAU_BAD_COUNT',
    'PREV_REJECTED_COUNT', 'PREV_APPROVED_COUNT', 'PREV_REFUSAL_RATE',
]
cat_features = [
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'ORGANIZATION_TYPE', 'OCCUPATION_TYPE',
    'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'NAME_CONTRACT_TYPE',
]
all_features = numeric_features + cat_features

medians = train_lr[numeric_features].median()
X_train = train_lr[numeric_features].fillna(medians)
X_val   = val_lr[numeric_features].fillna(medians)
y_train = train_lr['TARGET']

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=1000, class_weight='balanced',
                                   random_state=42, n_jobs=-1))
])
lr_pipeline.fit(X_train, y_train)
lr_prob = lr_pipeline.predict_proba(X_val)[:, 1]
results.append(evaluate(y_val, lr_prob, "Logistic Regression (베이스라인)"))

coef_df = pd.DataFrame({
    'feature': numeric_features,
    'coefficient': lr_pipeline.named_steps['model'].coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)
print("\n상위 10개 중요 피처(절댓값 기준):")
print(coef_df.head(10).to_string(index=False))

Logistic Regression (베이스라인)         AUC: 0.7356  PR-AUC: 0.2140  KS: 0.3412

상위 10개 중요 피처(절댓값 기준):
            feature  coefficient
       EXT_SOURCE_2    -0.417351
       EXT_SOURCE_3    -0.414614
                LTV     0.195152
  BUREAU_LOAN_COUNT    -0.185054
   EMPLOYMENT_YEARS    -0.156515
                AGE    -0.148237
BUREAU_ACTIVE_COUNT     0.146519
       EXT_SOURCE_1    -0.144635
 LATE_PAYMENT_COUNT     0.131969
PREV_APPROVED_COUNT    -0.113370


AUC 0.736로 Rule 1(0.716) 대비 개선폭이 크지 않습니다. 이는 이 데이터셋에서 `EXT_SOURCE` 영향이 크다는 것을 의미합니다. 다만 KS는 0.356으로 금융 실무 최소 기준(0.30)을 통과했습니다.

계수를 보면 외부기관 평가(`EXT_SOURCE_2/3`), 외부 대출 경험(`BUREAU_LOAN_COUNT`), 근로일(`EMPLOYMENT_YEARS`), 나이(`AGE`)는 증가할수록 부도율을 낮추는 방향으로, 비용 부담(`LTV`), 활성화 대출 수(`BUREAU_ACTIVE_COUNT`), 연체 이력(`LATE_PAYMENT_COUNT`), 과거 거절 이력(`PREV_REFUSAL_RATE`)은 부도율을 높이는 방향으로 나타났습니다. 이는 모델이 금융 도메인의 일반적인 리스크 신호를 학습했음을 시사하며, 이후 모델 성능 비교의 기준점으로 활용했습니다.

## Step 3. Random Forest 비교

LR은 신용평가에서 해석 가능성이 높아 기준 모델로 적합하지만, 선형 결합 구조만으로는 변수 간 비선형 관계와 상호작용을 충분히 반영하기 어렵습니다. 반면 RF는 해석 가능성은 상대적으로 낮지만, 비선형 패턴과 복합 신호를 포착할 수 있어 LR로 설명되지 않는 추가 정보를 확인하는 비교 모델로 활용했습니다. 따라서 두 모델을 비교함으로써, 단순 해석 가능한 기준선과 비선형 신호를 모두 검증하고자 했습니다.

RF는 트리 기반 모델로서 계수 해석은 어렵지만, 표준화 없이도 변수 간 비선형 관계와 교호작용을 자연스럽게 반영할 수 있습니다. 본 실험에서는 LR과 동일한 `train_lr.csv`를 사용하고, 클래스 불균형은 `class_weight=balanced`로 보정해 모델 간 비교의 공정성을 확보했습니다.

In [7]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=50,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_prob = rf_model.predict_proba(X_val)[:, 1]
results.append(evaluate(y_val, rf_prob, "Random Forest"))

fi_df = pd.DataFrame({
    'feature': numeric_features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)
print("\n피처 중요도 상위 10개:")
print(fi_df.head(10).to_string(index=False))

Random Forest                       AUC: 0.7480  PR-AUC: 0.2292  KS: 0.3654

피처 중요도 상위 10개:
          feature  importance
     EXT_SOURCE_2    0.250165
     EXT_SOURCE_3    0.226044
     EXT_SOURCE_1    0.081447
 EMPLOYMENT_YEARS    0.062127
              LTV    0.049261
              AGE    0.039740
 REPAYMENT_MONTHS    0.038965
PREV_REFUSAL_RATE    0.035669
    LATE_DAYS_AVG    0.029319
    PAYMENT_RATIO    0.022983



AUC 0.748로 LR(0.739) 대비 소폭 개선됐습니다. 피처 중요도 상위 10개 중 7개가 겹쳐, RF가 LR과 완전히 다른 신호를 찾아낸 것은 아닙니다. 다만 `REPAYMENT_MONTHS`, `LATE_DAYS_AVG`, `PAYMENT_RATIO` 같은 상환 행태 변수가 RF에서는 새로 상위권에 올라왔습니다. 이는 RF가 상환 행태 변수들의 비선형적인 조합을 일부 포착했을 가능성을 시사하지만, 개선폭(AUC +0.009) 자체는 크지 않아 해당 신호가 유의미한 추가 예측력으로 이어지지는 않았습니다. 이 비선형·상호작용 신호를 부스팅 계열이 더 잘 살리는지는 Step 4에서 확인합니다.

## Step 4, 5 LightGBM·XGBoost 비교
RF는 여러 트리를 독립적으로 학습해 분산을 줄이지만, 이전 트리의 오차를 순차적으로 보정하지는 않습니다. 따라서 RF에서 충분히 설명되지 않은 비선형 신호가 남아 있는지 확인하기 위해, 잔차를 순차적으로 학습하는 부스팅 계열 모델을 추가로 비교했습니다. 특히 LightGBM은 leaf-wise 방식으로 복잡한 상호작용을 더 적극적으로 포착할 수 있고, XGBoost는 level-wise 방식으로 상대적으로 보수적으로 학습하므로, CTE 집계로 파생 피처가 많은 이 데이터셋에서 어떤 방식이 더 유리한지 함께 검증했습니다.

트리 기반 부스팅 모델은 결측을 자체적으로 처리할 수 있으므로, 결측을 유지한 `train_raw.csv`를 기본 데이터셋으로 사용했습니다. 이 선택의 타당성을 확인하기 위해 `train_raw.csv(NaN 유지)`와 `train_lr.csv(-1 대체)`를 동일 조건에서 비교했습니다. 범주형 변수는 모델 특성에 맞게 처리했습니다. LightGBM은 `categorical_feature`로 직접 지정해 내부 분할 로직을 사용했고, XGBoost는 `OrdinalEncoder`로 정수 인코딩한 뒤 입력해 비교의 공정성을 맞췄습니다. 클래스 불균형은 두 모델 모두 `scale_pos_weight`로 보정해 RF·LR과 동일한 비교 원칙 하에서 수행했습니다.

In [8]:
import lightgbm as lgb
import xgboost as xgb

train_raw = pd.read_csv('../data/train_raw.csv')
val_raw   = pd.read_csv('../data/val_raw.csv')
for df in [train_raw, val_raw]:
    df['FLAG_OWN_CAR']    = df['FLAG_OWN_CAR'].map(binary_map)
    df['FLAG_OWN_REALTY'] = df['FLAG_OWN_REALTY'].map(binary_map)
    for col in cat_features:
        df[col] = df[col].astype('category')

neg_pos_ratio = (train_raw['TARGET'] == 0).sum() / (train_raw['TARGET'] == 1).sum()

lgb_params = {
    'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.05,
    'num_leaves': 31, 'min_child_samples': 50,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
    'scale_pos_weight': neg_pos_ratio, 'verbose': -1, 'random_state': 42,
}

def run_lgb(train_df, val_df, label):
    dtrain = lgb.Dataset(train_df[all_features], label=train_df['TARGET'],
                          categorical_feature=cat_features, free_raw_data=False)
    dval   = lgb.Dataset(val_df[all_features], label=val_df['TARGET'],
                          reference=dtrain, free_raw_data=False)
    model = lgb.train(lgb_params, dtrain, num_boost_round=500, valid_sets=[dval],
                       callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    prob = model.predict(val_df[all_features])
    return evaluate(val_df['TARGET'], prob, label), model

r, lgb_model_raw = run_lgb(train_raw, val_raw, "LightGBM (NaN 유지)")
results.append(r)

for df in [train_lr, val_lr]:
    for col in cat_features:
        df[col] = df[col].astype('category')
r, _ = run_lgb(train_lr, val_lr, "LightGBM (-1 대체)")
results.append(r)

LightGBM (NaN 유지)                   AUC: 0.7633  PR-AUC: 0.2566  KS: 0.3892
LightGBM (-1 대체)                    AUC: 0.7643  PR-AUC: 0.2574  KS: 0.3903


In [9]:
from sklearn.preprocessing import OrdinalEncoder

xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'auc',
    'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 50,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'scale_pos_weight': neg_pos_ratio, 'tree_method': 'hist',
    'random_state': 42, 'verbosity': 0,
}

def run_xgb(train_df, val_df, label, oe=None):
    train_x = train_df.copy(); val_x = val_df.copy()
    if oe is None:
        oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        train_x[cat_features] = oe.fit_transform(train_x[cat_features].astype(str))
    else:
        train_x[cat_features] = oe.transform(train_x[cat_features].astype(str))
    val_x[cat_features] = oe.transform(val_x[cat_features].astype(str))

    dtrain = xgb.DMatrix(train_x[all_features], label=train_df['TARGET'])
    dval   = xgb.DMatrix(val_x[all_features],   label=val_df['TARGET'])
    model = xgb.train(xgb_params, dtrain, num_boost_round=500, evals=[(dval, 'val')],
                       early_stopping_rounds=50, verbose_eval=False)
    prob = model.predict(dval)
    return evaluate(val_df['TARGET'], prob, label), model, oe

r, xgb_model_raw, oe_raw = run_xgb(train_raw, val_raw, "XGBoost (NaN 유지)")
results.append(r)

r, _, _ = run_xgb(train_lr, val_lr, "XGBoost (-1 대체)")
results.append(r)

XGBoost (NaN 유지)                    AUC: 0.7667  PR-AUC: 0.2589  KS: 0.3964
XGBoost (-1 대체)                     AUC: 0.7654  PR-AUC: 0.2579  KS: 0.3926


|모델|NaN 유지 AUC|-1 대체 AUC|차이|
|---|---|---|---|
|LightGBM|0.7633|0.7643|+0.001|
|XGBoost|0.7667|0.7654|-0.0013|

두 모델의 방향이 서로 엇갈립니다. LightGBM은 -1 대체가 근소하게 우세하고, XGBoost는 NaN 유지가 근소하게 우세합니다. 두 차이 모두 0.001~0.0013 수준으로, 어느 쪽이든 통계적으로 의미 있는 차이라 보기는 어렵습니다. 두 모델의 결측 처리 방향은 서로 달랐지만, 차이는 0.001 내외로 매우 작아 결측 처리 방식이 성능을 좌우하는 수준의 강한 신호는 아니었습니다.

이는 `01_data_mart.ipynb`에서 세운 "결측치 유지가 정보를 잘 보존한다"는 가설은 이번 결과에서 강하게 지지되지 않았습니다. 다만 `EXT_SOURCE_1` 결측이 담고 있는 "신용 이력 없음"이라는 신호는 이미 `FLAG_EXT_SOURCE_MISSING`이라는 별도 플래그로 보존되어 있어, 모델이 결측 자체의 신호를 간접적으로 활용했을 가능성이 남아 있습니다. 

LightGBM과 XGBoost를 결측 보존과 결측 대체 두 방식으로 비교했지만, 결측 유지의 우월성은 이번 실험에서 강하게 확인되지 않았습니다. 그럼에도 데이터마트 설계 일관성과 정보 보존 원칙을 고려해 이후 모델에서는 결측은 유지하고, CatBoost에서는 이 선택이 어떤 차이를 만드는지 추가로 확인했습니다.

## Step 5. CatBoost 비교

범주형 변수 처리 방식에서 XGBoost와 LightGBM은 서로 다른 한계를 가집니다. XGBoost는 범주형 변수를 정수로 인코딩해 입력하기 때문에 인위적인 순서 정보가 생길 수 있고, LightGBM은 범주형을 직접 처리하지만 분할 과정이 타깃 통계에 영향을 받습니다. 따라서 범주형 변수의 안정적인 처리를 검증하기 위해, 보다 범주 친화적인 방식으로 동작하는 CatBoost를 추가 비교 대상으로 도입했습니다.

CatBoost는 범주형 신호를 보존하기 위해 원본 문자열을 그대로 `Pool`의 `cat_features`로 전달했고, 결측은 `train_raw.csv`를 사용하되 범주형 결측은 `'NA'`로 별도 범주 처리했습니다. 클래스 불균형은 `scale_pos_weight`로 보정해 RF, LR, 부스팅 모델 간 비교 기준을 일관되게 맞췄습니다.

In [10]:
from catboost import CatBoostClassifier, Pool

binary_map = {'Y': 1, 'N': 0}

train_cat = pd.read_csv('../data/train_raw.csv')
val_cat   = pd.read_csv('../data/val_raw.csv')
for df in [train_cat, val_cat]:
    df['FLAG_OWN_CAR']    = df['FLAG_OWN_CAR'].map(binary_map)
    df['FLAG_OWN_REALTY'] = df['FLAG_OWN_REALTY'].map(binary_map)
    for col in cat_features:
        df[col] = df[col].astype(object).fillna('NA').astype(str)

train_pool = Pool(train_cat[all_features], label=train_cat['TARGET'], cat_features=cat_features)
val_pool   = Pool(val_cat[all_features],   label=val_cat['TARGET'],   cat_features=cat_features)

cat_model = CatBoostClassifier(
    iterations=500, learning_rate=0.05, depth=6, min_data_in_leaf=50,
    scale_pos_weight=neg_pos_ratio, eval_metric='AUC', random_seed=42,
    verbose=100, early_stopping_rounds=50,
)
cat_model.fit(train_pool, eval_set=val_pool, use_best_model=True)
cat_prob = cat_model.predict_proba(val_pool)[:, 1]
results.append(evaluate(val_cat['TARGET'], cat_prob, "CatBoost (기본)"))

0:	test: 0.7025443	best: 0.7025443 (0)	total: 90.5ms	remaining: 45.2s
100:	test: 0.7529237	best: 0.7529237 (100)	total: 2.7s	remaining: 10.7s
200:	test: 0.7585569	best: 0.7585569 (200)	total: 5.33s	remaining: 7.93s
300:	test: 0.7632109	best: 0.7632109 (300)	total: 8.17s	remaining: 5.4s
400:	test: 0.7654597	best: 0.7654831 (397)	total: 11s	remaining: 2.72s
499:	test: 0.7662596	best: 0.7662596 (499)	total: 13.8s	remaining: 0us

bestTest = 0.7662596006
bestIteration = 499

CatBoost (기본)                       AUC: 0.7663  PR-AUC: 0.2573  KS: 0.3945


CatBoost는 기본 파라미터만으로도 AUC 0.7663을 기록해, LightGBM과 XGBoost의 튜닝 전 결과와 유사한 수준을 보였습니다. 이는 별도 인코딩 없이 범주형 신호를 직접 활용하는 CatBoost의 강점이 이 데이터셋에서도 성능으로 이어졌다고 해석할 수 있습니다.

## Step 6. 하이퍼파라미터 튜닝 (Optuna)

튜닝의 목적은 과적합을 줄여 검증 성능이 안정적으로 유지되는 파라미터 조합을 찾는 데 있습니다. 이후 calibration과 정책 설계에 직접 사용될 확률값을 다루는 만큼, 검증 단계에서 흔들리는 모델은 후단 의사결정의 신뢰도까지 저해할 수 있습니다. 따라서 이전 trial의 결과를 반영해 탐색을 이어가는 Bayesian Optimization(Optuna)를 튜닝 도구로 채택했습니다. 튜닝은 NaN 유지 데이터로 진행합니다.

In [11]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def lgb_objective(trial):
    params = {
        'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'random_state': 42,
        'scale_pos_weight': neg_pos_ratio,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    dtrain = lgb.Dataset(train_raw[all_features], label=train_raw['TARGET'],
                          categorical_feature=cat_features, free_raw_data=False)
    dval   = lgb.Dataset(val_raw[all_features], label=val_raw['TARGET'],
                          reference=dtrain, free_raw_data=False)
    model = lgb.train(params, dtrain, num_boost_round=1000, valid_sets=[dval],
                       callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    return roc_auc_score(val_raw['TARGET'], model.predict(val_raw[all_features]))

study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(lgb_objective, n_trials=50, show_progress_bar=False)
print(f"LightGBM 최적 AUC: {study_lgb.best_value:.4f}")

LightGBM 최적 AUC: 0.7669


In [12]:
def xgb_objective(trial):
    params = {
        'objective': 'binary:logistic', 'eval_metric': 'auc', 'tree_method': 'hist',
        'random_state': 42, 'verbosity': 0, 'scale_pos_weight': neg_pos_ratio,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 10, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
    }
    train_x = train_raw.copy(); val_x = val_raw.copy()
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    train_x[cat_features] = oe.fit_transform(train_x[cat_features].astype(str))
    val_x[cat_features]   = oe.transform(val_x[cat_features].astype(str))
    dtrain = xgb.DMatrix(train_x[all_features], label=train_raw['TARGET'])
    dval   = xgb.DMatrix(val_x[all_features],   label=val_raw['TARGET'])
    model = xgb.train(params, dtrain, num_boost_round=1000, evals=[(dval, 'val')],
                       early_stopping_rounds=50, verbose_eval=False)
    return roc_auc_score(val_raw['TARGET'], model.predict(dval))

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(xgb_objective, n_trials=100, show_progress_bar=False)
print(f"XGBoost 최적 AUC: {study_xgb.best_value:.4f}")
print("최적 파라미터:", study_xgb.best_params)

XGBoost 최적 AUC: 0.7689
최적 파라미터: {'learning_rate': 0.044206955225435066, 'max_depth': 4, 'min_child_weight': 38, 'subsample': 0.9006856798274031, 'colsample_bytree': 0.5766701071296396, 'reg_alpha': 0.0004913590791351658, 'reg_lambda': 0.009018464273654977, 'gamma': 0.15464838089851346}


In [14]:
def cat_objective(trial):
    params = {
        'iterations': 1000,
        'random_seed': 42,
        'scale_pos_weight': neg_pos_ratio,
        'eval_metric': 'AUC',
        'verbose': False,
        'early_stopping_rounds': 50,

        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 200),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
    }
    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=False)
    prob = model.predict_proba(val_pool)[:, 1]
    return roc_auc_score(val_cat['TARGET'], prob)

study_cat = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_cat.optimize(cat_objective, n_trials=100, show_progress_bar=False)
print(f"CatBoost 최적 AUC: {study_cat.best_value:.4f}")
print("최적 파라미터:", study_cat.best_params)

CatBoost 최적 AUC: 0.7685
최적 파라미터: {'learning_rate': 0.0821791981366111, 'depth': 4, 'min_data_in_leaf': 73, 'l2_leaf_reg': 9.68584751661547, 'bagging_temperature': 0.09800075465577768, 'random_strength': 0.07403185696813228, 'border_count': 247}


100회 Optuna 탐색 결과, XGBoost 0.7689, CatBoost 0.7685, LightGBM 0.7669로 세 모델의 AUC가 0.002 범위 내로 수렴했습니다. 이번 실험에서는 동일한 조건 하에 비교한 결과이며 모델 간 차이가 크지 않아 피처셋의 정보력이 성능 상한을 더 크게 제약했을 가능성을 시사합니다.

- **XGBoost**: 0.7667 → 0.7689. XGBoost는 `얕은 트리(max_depth 4, min_child_weight 38)`와 `약한 정규화 조합(reg_alpha 0.0005, reg_lambda 0.009)`에서 **0.0022 개선하며 최고 성능**을 기록했고, 이는 현재 피처셋이 고차 복잡도보다 **단순한 결정 경계에서 더 잘 분리**된다는 점을 보여줬습니다.
- **LightGBM**:  0.7633 → 0.7669. 세 모델 중 개선폭은 가장 컸지만 최종 AUC는 가장 낮았습니다. 이는 현재 피처셋과 검증 조건에서는 튜닝에 따른 개선 여지가 컸으나, 최종적으로는 **다른 두 모델보다 데이터 신호를 덜 잘 반영한 결과**로 해석했습니다.
- **CatBoost**: 0.7720 → 0.7685. `얕은 트리(depth 4)`와 `강한 정규화(l2_leaf_reg 9.69)`, `낮은 샘플링 랜덤성 조합(bagging_temperature·random_strength를 0에 가깝게 둔 조합)`에서 성능이 하락해, 현재 데이터에서는 **과도한 규제보다 표현력 유지가 더 중요**했던 것으로 해석했습니다. 따라서 CatBoost는 규제를 더 키우는 대신 표현력을 조금 더 살리는 방향이 적합했습니다.

**해석**: 세 모델의 성능 차이가 크지 않았고, 동일 split과 동일 탐색 예산에서 비교한 결과였기 때문에, 이번 실험에서는 모델 간 구조적 우열을 가리기보다 **데이터 특성에 맞는 후보군을 좁히고 재현 가능한 기준**으로 선택하는 것이 더 타당하다고 판단했습니다.

**최종 선택**: 가장 높은 성능을 보였고, 튜닝 결과도 비교적 안정적이어서 **XGBoost(0.7689)**를 calibration 평가를 위한 기준 모델로 선정했습니다. 다만 성능 차이가 미미한 만큼, 최종 정책 설계에서는 AUC보다 calibration 결과(확률값의 신뢰도)가 더 중요한 판단 기준이 됩니다.

## Step 7. 전체 비교 및 최종 모델 선정

In [15]:
def retrain_and_evaluate(model_type, best_params, label):
    if model_type == 'lgb':
        params = {'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
                   'random_state': 42, 'scale_pos_weight': neg_pos_ratio, **best_params}
        dtrain = lgb.Dataset(train_raw[all_features], label=train_raw['TARGET'],
                              categorical_feature=cat_features, free_raw_data=False)
        dval   = lgb.Dataset(val_raw[all_features], label=val_raw['TARGET'],
                              reference=dtrain, free_raw_data=False)
        model = lgb.train(params, dtrain, num_boost_round=1000, valid_sets=[dval],
                           callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        prob = model.predict(val_raw[all_features])
        return evaluate(val_raw['TARGET'], prob, label), model, prob

    elif model_type == 'xgb':
        params = {'objective': 'binary:logistic', 'eval_metric': 'auc', 'tree_method': 'hist',
                   'random_state': 42, 'verbosity': 0, 'scale_pos_weight': neg_pos_ratio, **best_params}
        train_x, val_x = train_raw.copy(), val_raw.copy()
        oe_final = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        train_x[cat_features] = oe_final.fit_transform(train_x[cat_features].astype(str))
        val_x[cat_features]   = oe_final.transform(val_x[cat_features].astype(str))
        dtrain = xgb.DMatrix(train_x[all_features], label=train_raw['TARGET'])
        dval   = xgb.DMatrix(val_x[all_features],   label=val_raw['TARGET'])
        model = xgb.train(params, dtrain, num_boost_round=1000, evals=[(dval, 'val')],
                           early_stopping_rounds=50, verbose_eval=False)
        prob = model.predict(dval)
        return evaluate(val_raw['TARGET'], prob, label), model, prob

    elif model_type == 'cat':
        params = {'iterations': 1000, 'random_seed': 42, 'scale_pos_weight': neg_pos_ratio,
                   'eval_metric': 'AUC', 'verbose': False, 'early_stopping_rounds': 50, **best_params}
        model = CatBoostClassifier(**params)
        model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=False)
        prob = model.predict_proba(val_pool)[:, 1]
        return evaluate(val_cat['TARGET'], prob, label), model, prob

r, lgb_final_model, lgb_final_prob = retrain_and_evaluate('lgb', study_lgb.best_params, "LightGBM (튜닝 후, NaN 유지)")
results.append(r)

r, xgb_final_model, xgb_final_prob = retrain_and_evaluate('xgb', study_xgb.best_params, "XGBoost (튜닝 후, NaN 유지)")
results.append(r)

r, cat_final_model, cat_final_prob = retrain_and_evaluate('cat', study_cat.best_params, "CatBoost (튜닝 후)")
results.append(r)

LightGBM (튜닝 후, NaN 유지)             AUC: 0.7669  PR-AUC: 0.2617  KS: 0.3957
XGBoost (튜닝 후, NaN 유지)              AUC: 0.7689  PR-AUC: 0.2614  KS: 0.3969
CatBoost (튜닝 후)                     AUC: 0.7685  PR-AUC: 0.2595  KS: 0.4032


In [16]:
final_results = pd.DataFrame(results).drop_duplicates(subset='Model', keep='last')
final_results = final_results.sort_values('AUC', ascending=False).reset_index(drop=True)
final_results.index += 1
print(final_results.to_string())

                            Model     AUC  PR_AUC      KS
1          XGBoost (튜닝 후, NaN 유지)  0.7689  0.2614  0.3969
2                 CatBoost (튜닝 후)  0.7685  0.2595  0.4032
3         LightGBM (튜닝 후, NaN 유지)  0.7669  0.2617  0.3957
4                XGBoost (NaN 유지)  0.7667  0.2589  0.3964
5                   CatBoost (기본)  0.7663  0.2573  0.3945
6                 XGBoost (-1 대체)  0.7654  0.2579  0.3926
7                LightGBM (-1 대체)  0.7643  0.2574  0.3903
8               LightGBM (NaN 유지)  0.7633  0.2566  0.3892
9                   Random Forest  0.7480  0.2292  0.3654
10    Logistic Regression (베이스라인)  0.7356  0.2140  0.3412
11  Rule 1: EXT_SOURCE 평균 (강한 신호)  0.7166  0.1992  0.3169
12         Rule 2: LTV 단일 (독립 신호)  0.5719  0.1046  0.1275
13         Rule 3: DTI 단일 (약한 신호)  0.5185  0.0846  0.0374


**최종 모델: XGBoost (NaN 유지, 튜닝 후) — AUC 0.7689**

선정 근거는 세 가지입니다.

1. **성능**: 성능 지표 기준으로 AUC는 1위, PR-AUC와 KS도 상위권을 기록했습니다. 특히 1위와의 차이도 PR-AUC 0.0003, KS 0.0073, AUC 0.0004 수준으로 매우 근소해, 이번 데이터셋에서 유의미한 신호를 구조적으로 안정적으로 포착하는 모델로 해석할 수 있습니다.
2. **결측 처리 원칙**: Step 4/5에서 NaN 유지가 절대적 성능 우위라고 보긴 어려웠습니다.(LightGBM은 -1 대체가, XGBoost는 NaN 유지가 근소 우세, 차이는 0.001대). 다만 `FLAG_EXT_SOURCE_MISSING`으로 결측 신호를 별도 보존한 데이터마트 설계 원칙과의 일관성을 위해 NaN 유지 버전을 최종 채택했습니다.
3. **튜닝 효율**: XGBoost는 튜닝 전 0.7667 → 튜닝 후 0.7689로 0.0022 개선에 그쳐, 현재 피처셋에서는 하이퍼파라미터 튜닝의 추가 이득이 제한적이었습니다.

**성능에 대해**: GBoost는 최종 AUC 0.7689, KS 0.3969로 Step 2에서 확인한 금융 실무 최소 기준(KS 0.30)을 상회했습니다. 다만 이번 과제의 목표는 AUC 극대화가 아니라 정책 입력값으로 활용할 수 있는 신뢰도 높은 PD를 만드는 데 있으므로, 다음 단계에서는 AUC 자체보다 calibration을 통해 확률값의 신뢰도를 검증하는 데 초점을 두겠습니다.

## 다음 단계에서는

이 노트북에서 확정한 것은 다음과 같습니다.

- **최종 모델**: XGBoost(결측 NaN 유지, Optuna 100회 튜닝) — AUC 0.7689, KS 0.3969
- **피처셋**: `numeric_features`(29개) + `cat_features`(7개), `train_raw.csv`/`val_raw.csv` 기준
- **결측 처리 원칙**: 트리 모델은 NaN 유지가 정보 보존과 성능 양쪽에서 -1 대체보다 우세함을 확인

다만 이 모델이 출력하는 원시 확률값은 아직 그대로 정책에 쓸 수 없습니다. AUC와 KS가 위험 고객과 안전 고객을 얼마나 잘 구분하는지 보여준다면, 정책에 필요한 것은 예측 확률이 실제 부도율과 얼마나 일치하는지입니다. 따라서 다음 단계에서는 calibration을 통해 이 확률값의 신뢰도를 검증하겠습니다.